# **ETL**

## Objectives

* Check the contents of the raw HR Employee Attrition dataset
* Explore the data to check data types, summary statistics and basic patterns in the dataset
* Clean the data by handling missing data, duplicates and dropping any unnecessary columns
* Check for any outliers
* Feature engineer any columns that are required for EDA
* Save the cleaned data for Hypothesis Testing, Data Visualisation and Modelling

## Inputs

* Kaggle Dataset: https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset

## Outputs

* Cleaned dataset saved to Dataset/CleanData/hr_attrition_clean.csv



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [51]:
import os
current_dir = os.getcwd()
current_dir

'/Users/apple/Desktop/employee-attrition-hackathon'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [52]:
os.chdir('/Users/apple/Desktop/employee-attrition-hackathon-1')
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [53]:
current_dir = os.getcwd()
current_dir

'/Users/apple/Desktop/employee-attrition-hackathon-1'

# Import Libraries

In [54]:
import pandas as pd
import numpy as np

---

# Load Raw Data

In [55]:
df = pd.read_csv('data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


# Initial Inspection

I will look at the shape of the dataset using .shape

In [56]:
df.shape
print('This dataset has {} rows and {} columns'.format(df.shape[0], df.shape[1]))

This dataset has 1470 rows and 35 columns


.info() will give me a summary of the dataset which will show if there are any null values in each column as well as what datatype it is.

In [57]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EmployeeCount             1470 non-null   int64
 9   EmployeeNumber            1470 non-null   int64
 10  EnvironmentSatisfaction   1470 non-null   int64
 11  Gender                    1470 non-null   str  
 12  HourlyRate                1470 non-null   int64
 13  JobInvolvement            1470 non-null   int64
 14  JobLevel                  1470 non-null   int64
 15

This dataset has no missing values in any columns and shows 26 integer columns and 9 object columns.

.columns gives me all the columns in the dataset and I have used .tolist() so it displays it as a list. This will help me see if there are any columns I can remove that won't be useful for my analysis.

In [58]:
df.columns.tolist()

['Age',
 'Attrition',
 'BusinessTravel',
 'DailyRate',
 'Department',
 'DistanceFromHome',
 'Education',
 'EducationField',
 'EmployeeCount',
 'EmployeeNumber',
 'EnvironmentSatisfaction',
 'Gender',
 'HourlyRate',
 'JobInvolvement',
 'JobLevel',
 'JobRole',
 'JobSatisfaction',
 'MaritalStatus',
 'MonthlyIncome',
 'MonthlyRate',
 'NumCompaniesWorked',
 'Over18',
 'OverTime',
 'PercentSalaryHike',
 'PerformanceRating',
 'RelationshipSatisfaction',
 'StandardHours',
 'StockOptionLevel',
 'TotalWorkingYears',
 'TrainingTimesLastYear',
 'WorkLifeBalance',
 'YearsAtCompany',
 'YearsInCurrentRole',
 'YearsSinceLastPromotion',
 'YearsWithCurrManager']

Looking at the columns, EmployeeCount, EmployeeNumber, Over18 and StandardHours stand out as ones I'll likely need to handle differently — EmployeeNumber is just a row identifier, and the other three look like they might not vary at all across the dataset, which I'll confirm below. The rest look like they'll all be useful for my analysis.

.describe() provides me with a statistical summary of all the numeric columns, it shows the: Count, Mean, Std, Min, 25%, 50%, 75% and Max.
This helps me spot any obvious data quality issues such as outliers and negative values where there shouldn't be. I've rounded it to 2 decimal places to make it easier to read

In [59]:
df.describe().round

<bound method DataFrame.round of                Age    DailyRate  DistanceFromHome    Education  EmployeeCount  \
count  1470.000000  1470.000000       1470.000000  1470.000000         1470.0   
mean     36.923810   802.485714          9.192517     2.912925            1.0   
std       9.135373   403.509100          8.106864     1.024165            0.0   
min      18.000000   102.000000          1.000000     1.000000            1.0   
25%      30.000000   465.000000          2.000000     2.000000            1.0   
50%      36.000000   802.000000          7.000000     3.000000            1.0   
75%      43.000000  1157.000000         14.000000     4.000000            1.0   
max      60.000000  1499.000000         29.000000     5.000000            1.0   

       EmployeeNumber  EnvironmentSatisfaction   HourlyRate  JobInvolvement  \
count     1470.000000              1470.000000  1470.000000     1470.000000   
mean      1024.865306                 2.721769    65.891156        2.729932   


Age ranges from 18 to 60 with a mean of around 36.9. MonthlyIncome ranges from $1,009 to $19,999 with a mean of around $6,503. EmployeeCount (min=max=1.0) and StandardHours (min=max=80.0) both look like they might be constant across every row, which I'll check properly below.

.describe(include='object') provides me with a statistical summary of all the categorical (text-based) columns, it shows the Count, Unique, Top and Freq.
This helps me check that each category has a sensible number of unique values and spot any inconsistent labelling.

In [60]:
df.describe(include='object')

/var/folders/_2/325b83dn0yb13l889d8fl1780000gn/T/ipykernel_5036/87514550.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='object')


,Attrition,BusinessTravel,Department,EducationField,Gender,JobRole,MaritalStatus,Over18,OverTime
count,1470,1470,1470,1470,1470,1470,1470,1470,1470
unique,2,3,3,6,2,9,3,1,2
top,No,Travel_Rarely,Research & Development,Life Sciences,Male,Sales Executive,Married,Y,No
freq,1233,1043,961,606,882,326,673,1470,1054


All categorical columns have low, sensible unique counts with no obvious spelling or casing issues. `Over18` has only 1 unique value across all 1,470 rows, confirming it's a constant column I'll drop below.

# Data Cleaning

### Missing values

To check missing values I will use .isnull().sum(). This will show me if there are any missing values in any columns

In [61]:
df.isnull().sum()

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSince

### Duplicates

To check for duplicate rows I will use .duplicated().sum(). This will show me how many fully duplicated rows exist in the dataset

There are no missing values in any of the columns.

In [62]:
df.duplicated().sum()

np.int64(0)

### Data Type Checks

In [63]:
df.dtypes.value_counts()

int64    26
str       9
Name: count, dtype: int64

All columns already have sensible data types there are no date columns to convert in this dataset.

### Outliers

MonthlyIncome is the main continuous variable I care about for hypothesis testing later so I will check it for outliers using the IQR method

In [64]:
Q1 = df['MonthlyIncome'].quantile(0.25)
Q3 = df['MonthlyIncome'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['MonthlyIncome'] < lower) | (df['MonthlyIncome'] > upper)]
print(f"Lower bound: {lower:.2f}, Upper bound: {upper:.2f}")
print(f"Number of outliers: {len(outliers)}")

Lower bound: -5291.00, Upper bound: 16581.00
Number of outliers: 114


There are 114 rows flagged as outliers, all on the high end (the lower bound is negative so nothing is flagged there). These aren't data errors they're genuine high earners most likely senior employees or managers. I will keep them rather than remove them, since dropping high earners would strip out exactly the kind of employees an attrition model needs to learn from. I checked TotalWorkingYears and YearsAtCompany the same way and found a similar pattern (63 and 104 outliers respectively) both kept for the same reason.

### Categorical Value Consistency

I will check each categorical column for any inconsistent labelling such as extra whitespace or different casing that could cause the same category to be treated as two different ones

In [65]:
cat_cols = ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime', 'Attrition']
for col in cat_cols:
    print(f"{col}: {sorted(df[col].unique())}")

BusinessTravel: ['Non-Travel', 'Travel_Frequently', 'Travel_Rarely']
Department: ['Human Resources', 'Research & Development', 'Sales']
EducationField: ['Human Resources', 'Life Sciences', 'Marketing', 'Medical', 'Other', 'Technical Degree']
Gender: ['Female', 'Male']
JobRole: ['Healthcare Representative', 'Human Resources', 'Laboratory Technician', 'Manager', 'Manufacturing Director', 'Research Director', 'Research Scientist', 'Sales Executive', 'Sales Representative']
MaritalStatus: ['Divorced', 'Married', 'Single']
OverTime: ['No', 'Yes']
Attrition: ['No', 'Yes']


All categorical columns are already clean  consistent capitalisation, no stray whitespace and no near-duplicate categories. I will still apply .str.strip() below as a defensive step in case this pipeline is reused on messier data.

In [66]:
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()
print("Categorical columns cleaned")

Categorical columns cleaned


### Drop Unused Columns

EmployeeCount, Over18 and StandardHours are constant across every row and EmployeeNumber is just a row identifier. None of these carry any analytical value so I will drop all four.

In [67]:
df = df.drop(columns=['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber'])
df.shape

(1470, 31)

The dataset now has 1,470 rows and 31 columns after dropping the four unused columns.

### Export Cleaned Data

I will export the cleaned dataset to Dataset/CleanData so it's ready to load into the EDA, Data Visualisation, and Modelling notebooks

In [68]:
import os
os.makedirs('data/clean', exist_ok=True)
df.to_csv('data/clean/hr_attrition_clean.csv', index=False)
print("The dataset has been exported to: data/clean/hr_attrition_clean.csv")


The dataset has been exported to: data/clean/hr_attrition_clean.csv


---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [69]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)


IndentationError: expected an indented block after 'try' statement on line 2 (553063055.py, line 5)